In [1]:
import pandas as pd 


# 读取 Feather 文件
data = pd.read_feather(rf'/project/model_share/share_1_new/factor_csy/fac20250721/merged_data.fea')

print("✅ Feather 文件读取完成！")


✅ Feather 文件读取完成！


In [2]:
data[:10]

,date,Code,a0001_factor_fea_20250630,a0002_factor_fea_20250630,a0003_factor_fea_20250630,a0004_factor_fea_20250630,a0005_factor_fea_20250630,a0006_factor_fea_20250630,a0007_factor_fea_20250630,a0008_factor_fea_20250630,...,fac3035,fac3036,fac3037,fac3038,fac3039,fac3040,fac3041,fac3042,fac3043,fac3044
0,20210105,000001,0.438252,0.238076,-0.526136,0.809813,-0.496128,1.251059,-0.517327,-0.322449,...,-0.110591,0.050464,0.064927,0.081030,0.095168,0.101904,0.127329,0.043079,0.027992,-0.084919
1,20210105,000002,0.560824,0.345450,-0.000796,-0.190782,0.148574,1.324764,0.147143,-0.169908,...,-0.228317,-0.036660,-0.110984,-0.032141,-0.101185,-0.002097,-0.001209,0.002828,-0.094783,-0.222358
2,20210105,000004,0.124771,-0.716989,-0.245919,-0.690275,0.473380,-0.145338,1.134138,0.684091,...,0.055253,-0.046485,-0.041135,-0.077998,-0.073629,-0.206159,-0.227427,-0.225368,-0.180979,0.049022
3,20210105,000005,0.000000,-1.345319,-0.979040,-2.109435,-0.685095,0.000000,-1.176065,-1.347414,...,-0.109035,-0.070376,-0.042297,-0.065443,-0.039849,-0.052984,-0.050973,-0.050437,-0.062537,-0.094527
4,20210105,000006,-1.926162,-0.902478,-1.627878,-1.450586,-0.793003,-1.223660,-0.107143,-0.298837,...,-6.354914,-2.125352,-2.111854,-1.813738,-1.803210,0.516527,0.632701,1.607664,0.117812,-6.511986
5,20210105,000007,0.000000,0.000000,0.000000,0.000000,-0.958644,0.000000,0.000000,-0.592691,...,-0.689772,-0.476745,-0.534680,-0.442091,-0.496562,-0.095907,-0.093943,0.268501,-0.065188,-0.759542
6,20210105,000008,-1.674689,-1.610272,-2.190131,-0.919477,0.172273,-2.006026,1.535360,0.770608,...,-0.116423,-0.030849,-0.034073,-0.034950,-0.038188,-0.062829,-0.065853,-0.070390,-0.075561,-0.098238
7,20210105,000009,-0.291506,-0.845572,-1.447605,-0.690400,-0.175880,-1.104696,-0.284196,-0.292768,...,0.734418,-0.011135,-0.014088,-0.008762,-0.011626,-0.418605,-0.391069,-0.415001,-0.243183,0.628947
8,20210105,000010,-0.968894,-0.503371,-1.994338,-1.674059,1.643491,-1.685057,-0.175107,1.853964,...,0.028831,0.077215,0.099959,0.065808,0.086970,0.058103,0.047798,-0.015257,-0.002337,0.015200
9,20210105,000011,-0.493728,-0.608667,-0.737641,-0.979190,1.315775,0.141545,-0.366610,1.252828,...,-0.194839,-0.148061,-0.014604,-0.134712,-0.011944,-0.042732,-0.039240,-0.063735,-0.069914,-0.188551


In [3]:
data = data.set_index("date").sort_index()
date_list = data.index.unique().tolist()
Code_list_data = data["Code"].unique().tolist()

total_stock_count = len(Code_list_data)
total_date_count = len(date_list)

In [ ]:
stock_appearance_days = data.groupby("Code").size()  # 返回 Series：每只股票对应的行数（即出现的天数）
stocks_per_day = data.groupby("date")["Code"].nunique()  # 返回 Series：每个日期对应的不重复股票数

In [ ]:
stock_appearance_days[:10]
# stocks_per_day[-10:]

In [ ]:
stock_appearance_days_list = stock_appearance_days.to_list()
stocks_per_day_list = stocks_per_day.to_list()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(stock_appearance_days_list, bins=50)
plt.grid(True)
plt.show()


In [ ]:

plt.figure(figsize=(10, 6))
plt.plot(stock_appearance_days_list, marker='o', linestyle='-', color='b')
plt.show()



In [ ]:

plt.figure(figsize=(10, 6))
plt.hist(stocks_per_day_list, bins=50)
plt.grid(True)
plt.show()


In [ ]:

plt.figure(figsize=(10, 6))
plt.plot(stocks_per_day_list, marker='o', linestyle='-', color='b')
plt.show()

In [ ]:
good_code_index = stock_appearance_days[stock_appearance_days > 968].index.tolist()

In [ ]:
good_data = data[data["Code"].isin(good_code_index)]

In [ ]:
good_data[3600:3630]

In [ ]:
# 1. 获取所有唯一日期和股票代码
all_dates = good_data.index.unique()          # 因为你的 data 已设定 "date" 为 index
all_codes = good_data["Code"].unique()

# 2. 创建 MultiIndex：所有日期 × 所有股票
full_index = pd.MultiIndex.from_product([all_dates, all_codes], names=["date", "Code"])

# 3. 将原数据以 MultiIndex 形式重建，并 reindex 到完整索引上
good_data_full = good_data.reset_index().set_index(["date", "Code"]).reindex(full_index)


In [ ]:
# 1. 重置索引，把 "date" 和 "Code" 变回普通列
good_data_full_reset = good_data_full.reset_index()

# 2. 保存为 feather 格式（.fea 只是简称，实际上是 .feather 文件）
# good_data_full_reset.to_feather("/project/model_share/share_1_new/factor_csy/fac20250625_v2/fac20250625_v2.fea")